# ML Model — Demand Forecasting

**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Source:** `main.default.gold_ml_features`

## Goal
Predict `next_month_sales` for each `item_type + supplier` combination using lag features, rolling averages, and seasonality encoding.

## Notebook Structure

| Section | What it does |
|---------|-------------|
| **Setup & Imports** | Loads all required libraries (scikit-learn, LightGBM, XGBoost, MLflow, pandas, numpy) and prints package versions for reproducibility |
| **Load Data** | Reads the `gold_ml_features` Delta table from Unity Catalog and converts it to pandas — 8,219 rows, 14 columns, 0 nulls |
| **Feature Engineering** | Encodes categorical columns (`item_type`, `supplier_tier`) as integers using LabelEncoder and defines the 10 feature columns and the target variable `next_month_sales` |
| **Temporal Split** | Splits data by year — train set covers 2017–2019 (7,101 rows), test set covers 2020 (1,118 rows). Creates a `TimeSeriesSplit` object with 5 folds for cross-validation |
| **Evaluation Function** | Defines `evaluate_model()` — a reusable function that runs 5-fold cross-validation on the train set, trains on the full train set, predicts on the test set, and returns MAE, RMSE, and R² |
| **Model Training** | Trains 4 models (Linear Regression, Random Forest, LightGBM, XGBoost) and logs each run to MLflow with CV MAE, Test MAE, and Test R² |
| **Model Comparison** | Builds a comparison table with all metrics sorted by CV MAE. Identifies XGBoost as best on validation and LightGBM as best on test |
| **Feature Importance** | Extracts and formats feature importances from the best model by CV MAE. Shows that `lag_2_sales` (41.91%) and `lag_1_sales` (30.47%) drive 71% of all predictions |
| **Predictions vs Actual** | Scatter plot of LightGBM predictions vs actual values on the 2020 test set. Shows the top 5 largest errors — all BEER, all in March 2020 (COVID-19 demand shock) |
| **Monthly Aggregation** | Line plot of total actual vs predicted sales aggregated by month. Shows the model underestimates January (-15.41%) and March (-12.79%) and overestimates July (+7.31%) |
| **Breakdown by Item Type** | Calculates MAE and R² separately for each product category. Identifies LIQUOR and WINE as production-ready, KEGS as insufficient (R² = -0.04), and BEER errors as COVID-driven |
| **Breakdown by Supplier Tier** | Calculates MAE and R² per supplier tier. Shows `rest` has the lowest dollar error, `top3` has the strongest R², and `top15` is the hardest segment to forecast |
| **Model Registry** | Registers the trained LightGBM model in the MLflow Model Registry under `workspace.default.lightgbm_sales_forecaster`. Saves `le_item` and `le_tier` encoders as MLflow artifacts |
| **Inference Example** | Loads the registered model and encoders from MLflow and runs a full prediction on a new record — BEER, top3 supplier, January 2025. Predicted next month sales: $57,575.43 |
| **Results & Conclusions** | Full summary of model ranking, feature importance findings, segment analysis, monthly error patterns, and limitations |

## Models

| Model | Type | Why |
|-------|------|-----|
| Linear Regression | Baseline | Simplest model — sets the performance floor |
| Random Forest | Bagging | Multiple trees in parallel — robust, low variance |
| LightGBM | Gradient Boosting | Fast, memory-efficient, handles tabular data well |
| XGBoost | Optimized Boosting | Regularized boosting — industry standard for tabular data |

## Train / Test Split

| Set | Years | Rows |
|-----|-------|------|
| Train + CV | 2017, 2018, 2019 — TimeSeriesSplit (5 folds) | 7,101 |
| Test | 2020 — touched once, final evaluation only | 1,118 |

**TimeSeriesSplit — 5 folds over train set:**

| Fold | Train rows | Validation rows |
|------|------------|-----------------|
| 1 | 1,186 | 1,183 |
| 2 | 2,369 | 1,183 |
| 3 | 3,552 | 1,183 |
| 4 | 4,735 | 1,183 |
| 5 | 5,918 | 1,183 |

## Evaluation Metrics

| Metric | What it measures |
|--------|-----------------|
| MAE | Average dollar error — easy to interpret |
| RMSE | Penalizes large errors more heavily — detects big failures |
| R² | Percentage of variance explained by the model |

## Note on Test Set

2020 data is incomplete (3 non-consecutive months: January, March, July) and coincides with the COVID-19 pandemic — a global demand shock that no model trained on 2017–2019 data could reasonably predict.

This is intentional. Keeping 2020 as the test set produces honest metrics and demonstrates how the model behaves under conditions it was never trained on.

## Setup & Imports

Libraries used in this notebook:
- `scikit-learn` — Linear Regression, Random Forest, TimeSeriesSplit, metrics
- `lightgbm` — gradient boosting (LightGBM)
- `xgboost` — gradient boosting (XGBoost)
- `mlflow` — experiment tracking and model logging

In [0]:
# SETUP & IMPORTS
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries loaded")

## Load Data

Loads `gold_ml_features` from the Delta table built in notebook `04_gold_layer`.

- 8,219 rows
- 14 columns
- Years: 2017, 2018, 2019, 2020
- Zero null values

In [0]:
# LOAD DATA FROM GOLD TABLE
df_spark = spark.table("main.default.gold_ml_features")
df = df_spark.toPandas()

print(f"Shape       : {df.shape}")
print(f"Columns     : {list(df.columns)}")
print(f"Years       : {sorted(df['year'].unique())}")
print(f"Null values : {df.isnull().sum().sum()}")

## Feature Engineering & Encoding

Categorical columns (`item_type`, `supplier_tier`) are encoded as integers using `LabelEncoder` since ML algorithms only accept numeric input.

**Features (10):** `item_type_enc`, `supplier_tier_enc`, `lag_1_sales`, `lag_2_sales`, `rolling_3m_avg`, `warehouse_ratio`, `transaction_count`, `month_sin`, `month_cos`, `year`

**Target:** `next_month_sales`

In [0]:
# FEATURE ENGINEERING & ENCODING

# LabelEncoder converts text categories to integers
# Example: "BEER" → 0, "KEGS" → 1, "LIQUOR" → 2
le_item = LabelEncoder()
le_tier  = LabelEncoder()

# fit_transform() learns the mapping and applies it in one step
df["item_type_enc"]     = le_item.fit_transform(df["item_type"])
df["supplier_tier_enc"] = le_tier.fit_transform(df["supplier_tier"])

# The 10 features the model will use to predict next month sales
FEATURE_COLS = [
    "item_type_enc",       # Product category encoded as integer
    "supplier_tier_enc",   # Supplier size encoded as integer (top3/top15/rest)
    "lag_1_sales",         # Sales from 1 month ago
    "lag_2_sales",         # Sales from 2 months ago
    "rolling_3m_avg",      # Average sales of the last 3 months
    "warehouse_ratio",     # Proportion of sales through warehouse vs retail
    "transaction_count",   # Number of transactions that month
    "month_sin",           # Seasonality — sine component
    "month_cos",           # Seasonality — cosine component
    "year",                # Year trend
]

# Target variable — what the model must learn to predict
TARGET_COL = "next_month_sales"

print(f"Features : {len(FEATURE_COLS)}")
print(f"Target   : {TARGET_COL}")
print(f"\nFeature dtypes:\n{df[FEATURE_COLS].dtypes}")

## Temporal Split

Data is split by year to respect the natural order of time:

| Set | Years | Rows |
|-----|-------|------|
| Train + CV | 2017, 2018, 2019 | 7,101 |
| Test | 2020 | 1,118 |

`TimeSeriesSplit` with 5 folds is used for cross-validation — validation always uses future data relative to training. Standard `train_test_split` is not used because it would allow training on future data and validating on the past.

> **Note on test set:** 2020 data is incomplete and coincides with the COVID-19 pandemic — a demand shock no model trained on 2017–2019 could reasonably predict. This is intentional. Keeping 2020 as the test set produces honest metrics and shows how the model behaves under conditions it was never trained on.

In [0]:
# TEMPORAL SPLIT

# Train set: 2017, 2018, 2019 — the model learns from this data
train_mask   = df["year"].isin([2017, 2018, 2019])
X_train_full = df.loc[train_mask, FEATURE_COLS].values
y_train_full = df.loc[train_mask, TARGET_COL].values

# Test set: 2020 — locked until final evaluation only
test_mask = df["year"] == 2020
X_test    = df.loc[test_mask, FEATURE_COLS].values
y_test    = df.loc[test_mask, TARGET_COL].values

# TimeSeriesSplit ensures validation always uses future data
# Never trains on future and validates on past
tscv = TimeSeriesSplit(n_splits=5)

print("=" * 45)
print("  TEMPORAL SPLIT SUMMARY")
print("=" * 45)
print(f"  Train+CV rows : {len(X_train_full):,}")
print(f"  Test rows     : {len(X_test):,}")
print(f"  CV folds      : 5 (TimeSeriesSplit)")
print("=" * 45)

print("\nFold breakdown:")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_full), 1):
    print(f"  Fold {fold} → train: {len(train_idx):,} | val: {len(val_idx):,}")

## Evaluation Helper Function

`evaluate_model()` centralizes all evaluation logic to avoid repeating the same code across the four models.

For each model it runs two evaluations:

**1. Cross-validation on the train set (2017–2019)**
Splits the train set into 5 folds using `TimeSeriesSplit` and reports the average MAE across all folds, plus the standard deviation to measure consistency.

**2. Final evaluation on the test set (2020)**
Trains the model on the full train set, predicts on 2020 data, and computes MAE, RMSE, and R².

Note: `cross_val_score` returns negative MAE by sklearn convention — it is multiplied by -1 to return positive values.

**Metrics returned:**

| Metric | Description |
|--------|-------------|
| CV MAE | Average dollar error across the 5 validation folds |
| CV MAE ± | Standard deviation of the error across folds — measures consistency |
| Test MAE | Average dollar error on 2020 data — the honest real-world metric |
| Test RMSE | Same as MAE but large errors are penalized more heavily |
| Test R² | Percentage of sales variance explained by the model |

In [0]:
# EVALUATION HELPER FUNCTION

def evaluate_model(name, model, X_train, y_train, X_test, y_test, tscv):

    # cross_val_score returns negative MAE by convention in sklearn
    # We multiply by -1 to get positive values
    cv_mae_scores = cross_val_score(
        model, X_train, y_train,
        cv=tscv,
        scoring="neg_mean_absolute_error"
    )
    cv_mae     = -cv_mae_scores.mean()  # average across 5 folds
    cv_mae_std =  cv_mae_scores.std()   # how much it varies fold to fold

    # Train on the full train set, then predict on test
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate the three evaluation metrics
    test_mae  = mean_absolute_error(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_r2   = r2_score(y_test, y_pred)

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  CV MAE (train)  : {cv_mae:,.2f} ± {cv_mae_std:,.2f}")
    print(f"  Test MAE (2020) : {test_mae:,.2f}")
    print(f"  Test RMSE       : {test_rmse:,.2f}")
    print(f"  Test R²         : {test_r2:.4f}")
    print(f"{'='*50}")

    # Return all metrics as a dict for the comparison table
    return {
        "model_name":   name,
        "cv_mae":       round(cv_mae, 2),
        "cv_mae_std":   round(cv_mae_std, 2),
        "test_mae":     round(test_mae, 2),
        "test_rmse":    round(test_rmse, 2),
        "test_r2":      round(test_r2, 4),
        "model_object": model
    }

print("✓ evaluate_model() defined")

## Model Training

Four models are trained and tracked with MLflow:

| Model | Type | Reason |
|-------|------|--------|
| Linear Regression | Baseline | Sets the performance floor |
| Random Forest | Bagging | Multiple trees in parallel, robust to overfitting |
| LightGBM | Gradient Boosting | Fast, memory-efficient, handles tabular data well |
| XGBoost | Gradient Boosting | Regularized boosting, industry standard for tabular data |

Each run is logged to the MLflow experiment `warehouse-retail-sales-ml-experiments`

In [0]:
# CELL 6 — TRAIN ALL MODELS WITH MLFLOW TRACKING

mlflow.set_experiment("/Users/santilopbla06@gmail.com/warehouse-retail-sales-ml-experiments")

results = []

# 1. Linear Regression
# Simplest model — used as baseline to set the performance floor
with mlflow.start_run(run_name="LinearRegression"):
    r = evaluate_model(
        name   = "Linear Regression",
        model  = LinearRegression(),
        X_train=X_train_full, y_train=y_train_full,
        X_test =X_test,       y_test =y_test,
        tscv   =tscv
    )
    mlflow.log_metric("cv_mae",   r["cv_mae"])
    mlflow.log_metric("test_mae", r["test_mae"])
    mlflow.log_metric("test_r2",  r["test_r2"])
    results.append(r)

# 2. Random Forest
# Trains 200 trees in parallel and averages their predictions
# max_depth=8 and min_samples_leaf=5 prevent overfitting
with mlflow.start_run(run_name="RandomForest"):
    r = evaluate_model(
        name   = "Random Forest",
        model  = RandomForestRegressor(
            n_estimators    =200,  # number of trees
            max_depth       =8,    # max depth per tree
            min_samples_leaf=5,    # each leaf needs at least 5 rows
            random_state    =42
        ),
        X_train=X_train_full, y_train=y_train_full,
        X_test =X_test,       y_test =y_test,
        tscv   =tscv
    )
    mlflow.log_metric("cv_mae",   r["cv_mae"])
    mlflow.log_metric("test_mae", r["test_mae"])
    mlflow.log_metric("test_r2",  r["test_r2"])
    results.append(r)

# 3. LightGBM
# Builds trees sequentially, each one correcting the errors of the previous
# learning_rate=0.05 means small conservative steps — more stable training
with mlflow.start_run(run_name="LightGBM"):
    r = evaluate_model(
        name   = "LightGBM",
        model  = lgb.LGBMRegressor(
            n_estimators =300,   # number of boosting rounds
            learning_rate=0.05,  # step size per round
            max_depth    =6,     # max depth per tree
            num_leaves   =31,    # controls tree complexity
            random_state =42,
            verbose      =-1     # suppress training output
        ),
        X_train=X_train_full, y_train=y_train_full,
        X_test =X_test,       y_test =y_test,
        tscv   =tscv
    )
    mlflow.log_metric("cv_mae",   r["cv_mae"])
    mlflow.log_metric("test_mae", r["test_mae"])
    mlflow.log_metric("test_r2",  r["test_r2"])
    results.append(r)

# 4. XGBoost
# Same boosting logic as LightGBM but with stronger regularization
# subsample and colsample_bytree add randomness to reduce overfitting
with mlflow.start_run(run_name="XGBoost"):
    r = evaluate_model(
        name   = "XGBoost",
        model  = xgb.XGBRegressor(
            n_estimators    =300,  # number of boosting rounds
            learning_rate   =0.05, # step size per round
            max_depth       =6,    # max depth per tree
            subsample       =0.8,  # use 80% of rows per tree
            colsample_bytree=0.8,  # use 80% of features per tree
            random_state    =42,
            verbosity       =0     # suppress training output
        ),
        X_train=X_train_full, y_train=y_train_full,
        X_test =X_test,       y_test =y_test,
        tscv   =tscv
    )
    mlflow.log_metric("cv_mae",   r["cv_mae"])
    mlflow.log_metric("test_mae", r["test_mae"])
    mlflow.log_metric("test_r2",  r["test_r2"])
    results.append(r)

print("\n✓ All models trained and logged to MLflow")

## Model Comparison

Results sorted by CV MAE (lower = better).

| Model | CV MAE | CV MAE ± | Test MAE | Test RMSE | Test R² |
|-------|--------|----------|----------|-----------|---------|
| XGBoost | 116.80 | 61.10 | 305.23 | 1,820.86 | 0.9393 |
| Random Forest | 118.18 | 57.73 | 284.64 | 1,531.97 | 0.9570 |
| LightGBM | 255.03 | 254.55 | 276.81 | 1,312.87 | 0.9684 |
| Linear Regression | 346.56 | 245.58 | 359.67 | 2,038.51 | 0.9239 |

**What each metric means in business terms:**

- **CV MAE** — how wrong the model is on average, in dollars, when tested on data it has never seen during training. Lower is better. XGBoost averages $116.80 of error per monthly prediction during cross-validation.
- **CV MAE ±** — how consistent the model is across different time periods. A high value means the model is reliable in some months but unreliable in others. LightGBM's ±$254.55 means its accuracy varies significantly depending on the time period evaluated.
- **Test MAE** — the real-world error on 2020 data, which the model never saw during training. This is the most honest metric. LightGBM predicts monthly sales with an average error of $276.81 per supplier and product combination.
- **Test RMSE** — similar to MAE but large errors are penalized more heavily. A much higher RMSE than MAE indicates the model occasionally makes very large mistakes. LightGBM's RMSE of $1,312.87 suggests some months have significantly larger errors than the $276.81 average.
- **Test R²** — the percentage of sales variation the model can explain. LightGBM's R² of 0.9684 means it explains 96.84% of all sales fluctuations across suppliers and product types.

**Key finding:** XGBoost wins on CV MAE but LightGBM wins on Test MAE and R². This gap suggests XGBoost overfits slightly to the training distribution. LightGBM generalizes better to unseen data.

In [0]:
# CELL 7 — FINAL COMPARISON TABLE

# Build a dataframe with all metrics from every model
comparison_df = pd.DataFrame([{
    "Model"    : r["model_name"],
    "CV MAE"   : r["cv_mae"],
    "CV MAE ±" : r["cv_mae_std"],
    "Test MAE" : r["test_mae"],
    "Test RMSE": r["test_rmse"],
    "Test R²"  : r["test_r2"],
} for r in results])

# Sort by CV MAE — best model on validation data appears first
comparison_df = comparison_df.sort_values("CV MAE")

print("\nMODEL COMPARISON — sorted by CV MAE (lower = better)")
print(comparison_df.to_string(index=False))

# Identify the best model by CV MAE
best = comparison_df.iloc[0]
print(f"\n  Best model (CV) : {best['Model']}")
print(f"  CV MAE          : {best['CV MAE']:,.2f}")
print(f"  Test MAE        : {best['Test MAE']:,.2f}")
print(f"  Test R²         : {best['Test R²']:.4f}")

display(comparison_df)

## Feature Importance

Feature importances from the best model by CV MAE (XGBoost).

**Key finding:** `lag_2_sales` and `lag_1_sales` together account for ~71% of all model decisions. The strongest predictor of next month's sales is recent sales history. `supplier_tier` ranks third, ahead of the rolling average.

Seasonality (`month_sin`, `month_cos`), product type, and warehouse ratio contribute less than 5% combined — suggesting that for this dataset, recent momentum matters far more than calendar patterns or channel mix.

In [0]:
# CELL 8 — FEATURE IMPORTANCE

# Get the best model by CV MAE from the comparison table
best_model_name = comparison_df.iloc[0]["Model"]
best_result     = next(r for r in results if r["model_name"] == best_model_name)
best_model      = best_result["model_object"]

# Tree-based models expose feature_importances_ after training
# Linear models use coefficients instead
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature"   : FEATURE_COLS,
        "importance": best_model.feature_importances_
    })

    # Convert raw importance to percentage
    importance_df["importance_%"] = (importance_df["importance"] / importance_df["importance"].sum() * 100).round(2)

    # Sort by percentage descending and drop raw importance
    importance_df = importance_df.sort_values("importance_%", ascending=False).drop(columns="importance")

    # Format as string with % symbol so Databricks displays it cleanly
    importance_df["importance_%"] = importance_df["importance_%"].apply(lambda x: f"{x:.2f}%")

    print(f"Feature importance — {best_model_name}:")
    display(importance_df)

else:
    # For linear regression: use absolute value of coefficients
    coef_df = pd.DataFrame({
        "feature"    : FEATURE_COLS,
        "coefficient": best_model.coef_
    }).sort_values("coefficient", key=abs, ascending=False)

    print(f"Coefficients — {best_model_name}:")
    display(coef_df)

## Predictions vs Actual Values

Visual evaluation of LightGBM's predictions against real 2020 sales data.

Each point in the scatter plot represents one monthly prediction for a specific supplier and product type combination. Points close to the dashed line are accurate predictions — the further a point is from the line, the larger the error.

The 5 worst predictions are shown below the chart to identify which segments the model struggles with most.

## Error Analysis

The scatter plot shows that most predictions follow the perfect prediction line closely, confirming that LightGBM captures the general sales pattern well across suppliers and product types.

However, the model consistently underestimates high-volume sales — points in the upper range deviate above the line, meaning actual sales were higher than predicted. This is expected behavior: extreme sales months are rare in the training data, so the model has limited examples to learn from.

**The 5 largest errors reveal a clear pattern:**

All 5 worst predictions are BEER, all involve top3 or top15 suppliers, and 4 out of 5 occur in March 2020 — the month COVID-19 lockdowns began in the United States. Bars, restaurants, and retail locations closed abruptly, causing a demand shock that no model trained on 2017–2019 data could anticipate.

This is not a model failure. It is an external event that falls outside the scope of what any forecasting model can reasonably predict without real-time signals. It reinforces why 2020 was kept as the test set — to produce honest metrics under real-world conditions rather than ideal ones.

In [0]:
#   CELL 9 — PREDICTIONS VS ACTUAL VALUES (LightGBM)

import matplotlib.pyplot as plt

# Get LightGBM model from results
lgbm_result = next(r for r in results if r["model_name"] == "LightGBM")
lgbm_model  = lgbm_result["model_object"]

# Generate predictions on test set
y_pred_lgbm = lgbm_model.predict(X_test)

# Build a dataframe with actual vs predicted values
pred_df = df.loc[df["year"] == 2020, ["year", "month", "item_type", "supplier_tier"]].copy()
pred_df["actual"]    = y_test
pred_df["predicted"] = y_pred_lgbm.round(2)
pred_df["error"]     = (pred_df["predicted"] - pred_df["actual"]).round(2)
pred_df["abs_error"] = pred_df["error"].abs().round(2)

# Plot actual vs predicted
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(pred_df["actual"], pred_df["predicted"],
           alpha=0.4, s=15, color="#1D9E75")

# Perfect prediction line
max_val = max(pred_df["actual"].max(), pred_df["predicted"].max())
ax.plot([0, max_val], [0, max_val],
        color="#D85A30", linewidth=1.5, linestyle="--", label="Perfect prediction")

ax.set_xlabel("Actual sales (USD)")
ax.set_ylabel("Predicted sales (USD)")
ax.set_title("LightGBM — Predicted vs Actual Sales (Test set 2020)")
ax.legend()
plt.tight_layout()
plt.show()

# Show the 5 worst predictions
print("\nTop 5 largest errors:")
display(pred_df.sort_values("abs_error", ascending=False).head(5)[
    ["item_type", "supplier_tier", "month", "actual", "predicted", "error"]
])

## Monthly Sales: Actual vs Predicted (2020)

The scatter plot shows overall prediction accuracy across all records. This chart zooms out to show the monthly aggregate — total actual sales vs total predicted sales for each month available in the 2020 test set.

The test set covers only 3 months — January, March, and July — due to the incomplete nature of the 2020 data.

This view makes it possible to identify in which specific months the model struggles most and whether the errors follow a consistent pattern.

### Findings

The test set covers only 3 months of 2020 — January, March, and July — due to the incomplete nature of the dataset for that year.

The model underestimates sales in January (-15.41%) and March (-12.79%), and overestimates in July (+7.31%). The January and March errors reflect the COVID-19 demand shock — a surge in anticipatory buying followed by business closures that the model had no signal to predict. The July overestimation suggests the model expected a recovery that did not fully materialize, as many businesses remained partially closed through mid-2020.

In [0]:
# CELL 10 — MONTHLY SALES: ACTUAL VS PREDICTED (2020)

# Aggregate actual and predicted sales by month
monthly_df = pred_df.groupby("month").agg(
    actual    = ("actual",    "sum"),
    predicted = ("predicted", "sum")
).reset_index()

# Map month numbers to names for display
month_names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun",
               7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"}
monthly_df["month_name"] = monthly_df["month"].map(month_names)

# Plot actual vs predicted by month
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(monthly_df["month_name"], monthly_df["actual"],
        marker="o", linewidth=2, label="Actual", color="#1D9E75")

ax.plot(monthly_df["month_name"], monthly_df["predicted"],
        marker="s", linewidth=2, label="Predicted", color="#D85A30", linestyle="--")

ax.set_xlabel("Month (2020)")
ax.set_ylabel("Total Sales (USD)")
ax.set_title("Monthly Sales — Actual vs Predicted (Test set 2020)")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

# Print monthly breakdown
print("\nMonthly breakdown:")
monthly_df["error ($)"] = (monthly_df["predicted"] - monthly_df["actual"]).round(2)
monthly_df["error (%)"] = ((monthly_df["error ($)"] / monthly_df["actual"]) * 100).round(2)
display(monthly_df)

## Error Breakdown by Item Type

Overall metrics like MAE and R² describe average model performance across all products and suppliers. But averages hide important differences — the model may perform well on WINE and poorly on BEER, or vice versa.

This breakdown shows Test MAE and R² for each product category separately, making it possible to identify which segments the model predicts reliably and which ones it struggles with.

### Findings

| Segment | Behavior |
|---------|----------|
| STR_SUPPLIES | Most accurate in dollar terms — low and stable sales volume |
| KEGS | R² of -0.0418 — the model performs worse than predicting the average. Keg demand is driven by events and seasonal patterns that lag features cannot capture |
| NON-ALCOHOL | R² of 0.4968 — the model explains only 49% of variance. Niche products with irregular purchasing behavior |
| WINE | Moderate performance — reasonable MAE of $145.36 and R² of 0.9077 |
| LIQUOR | Good performance — MAE of $172.84 and R² of 0.9542 |
| BEER | Highest MAE in dollar terms ($907.23) but strongest R² (0.9688). Large errors are driven by the COVID-19 demand shock in March 2020 — not by systematic model failure. The underlying sales pattern is well captured |

**Key takeaway:** The model is production-ready for LIQUOR and WINE. For BEER it is reliable under normal conditions but sensitive to external shocks. For KEGS and NON-ALCOHOL, the current feature set is insufficient — additional signals such as event calendars or seasonal indicators would be needed to improve accuracy in those segments.

In [0]:
# CELL 10 — ERROR BREAKDOWN BY ITEM TYPE

# Rebuild test dataframe with predictions from all models
test_df = df.loc[df["year"] == 2020].copy().reset_index(drop=True)

# Add LightGBM predictions (best model)
test_df["predicted"] = lgbm_model.predict(X_test).round(2)
test_df["actual"]    = y_test
test_df["abs_error"] = (test_df["predicted"] - test_df["actual"]).abs()

# Calculate MAE and R² per item_type
breakdown = []

for item in test_df["item_type"].unique():
    subset = test_df[test_df["item_type"] == item]
    
    mae  = mean_absolute_error(subset["actual"], subset["predicted"])
    rmse = np.sqrt(mean_squared_error(subset["actual"], subset["predicted"]))
    r2   = r2_score(subset["actual"], subset["predicted"])
    
    breakdown.append({
        "item_type"  : item,
        "rows"       : len(subset),
        "MAE ($)"    : round(mae, 2),
        "RMSE ($)"   : round(rmse, 2),
        "R²"         : round(r2, 4),
    })

breakdown_df = pd.DataFrame(breakdown).sort_values("MAE ($)")

print("LightGBM — Test performance by item type:")
display(breakdown_df)

## Error Breakdown by Supplier Tier

The previous breakdown showed which product categories the model struggles with. This breakdown answers a different question — does the model perform equally well across supplier sizes?

`top3` suppliers have the highest sales volume and the most consistent historical patterns, so the model should predict them more accurately. `rest` suppliers have irregular, low-volume sales that are harder to forecast.

### Findings

| Supplier tier | Behavior |
|---------------|----------|
| rest | Lowest MAE in dollar terms ($112.22) — small suppliers have low and stable sales volumes that are easy to predict |
| top15 | Weakest R² (0.8751) — mid-size suppliers combine moderate sales volume with high variability, making them the hardest segment to forecast reliably |
| top3 | Highest MAE in dollar terms ($3,160.43) but strongest R² (0.9727) — the three largest suppliers generate the highest sales volumes, so absolute errors are large, but the model correctly captures their underlying sales pattern |

**Key takeaway:** MAE alone is misleading when comparing segments with very different sales volumes. A $3,160 error on a $99,000 sale is proportionally smaller than a $112 error on a $500 sale. R² is the more reliable metric for cross-segment comparison — and by that measure, top3 suppliers are the best predicted segment in the dataset.

In [0]:
# CELL 12 — ERROR BREAKDOWN BY SUPPLIER TIER

breakdown_tier = []

for tier in test_df["supplier_tier"].unique():
    subset = test_df[test_df["supplier_tier"] == tier]

    mae  = mean_absolute_error(subset["actual"], subset["predicted"])
    rmse = np.sqrt(mean_squared_error(subset["actual"], subset["predicted"]))
    r2   = r2_score(subset["actual"], subset["predicted"])

    breakdown_tier.append({
        "supplier_tier": tier,
        "rows"         : len(subset),
        "MAE ($)"      : round(mae, 2),
        "RMSE ($)"     : round(rmse, 2),
        "R²"           : round(r2, 4),
    })

breakdown_tier_df = pd.DataFrame(breakdown_tier).sort_values("MAE ($)")

print("LightGBM — Test performance by supplier tier:")
display(breakdown_tier_df)

## Model Registry & Encoder Persistence

After identifying LightGBM as the best model, two artifacts are saved to ensure the pipeline is reproducible and ready for inference on new data.

**Model registry (MLflow)**
The trained LightGBM model is logged to the MLflow Model Registry under the name `lightgbm_sales_forecaster`. This allows the model to be loaded and used for predictions without retraining.

**Encoder persistence (pickle)**
The two `LabelEncoder` objects (`le_item` and `le_tier`) are saved to disk. These encoders must be used on any new data before passing it to the model — without them, categorical columns cannot be transformed the same way they were during training, and predictions will be incorrect.

In [0]:
# CELL 11 — MODEL REGISTRY & ENCODER PERSISTENCE

import pickle
import tempfile

# Register LightGBM as the best model in MLflow Model Registry
with mlflow.start_run(run_name="LightGBM_best_model"):

    # Prepare an input example for signature inference
    # Use the first row of training data
    input_example = X_train_full[:1]

    # Log the model as a reusable artifact with signature
    mlflow.sklearn.log_model(
        sk_model       = lgbm_model,
        artifact_path  = "lightgbm_sales_forecaster",
        registered_model_name = "lightgbm_sales_forecaster",
        input_example  = input_example
    )

    # Log best model metrics for reference
    mlflow.log_metric("test_mae",  lgbm_result["test_mae"])
    mlflow.log_metric("test_rmse", lgbm_result["test_rmse"])
    mlflow.log_metric("test_r2",   lgbm_result["test_r2"])

    print("✓ Model registered in MLflow Model Registry")

    # Save encoders as MLflow artifacts
    # Use tempfile to create a writable temporary directory
    with tempfile.TemporaryDirectory() as tmpdir:
        # Save encoders to temp directory
        with open(f"{tmpdir}/le_item.pkl", "wb") as f:
            pickle.dump(le_item, f)
        
        with open(f"{tmpdir}/le_tier.pkl", "wb") as f:
            pickle.dump(le_tier, f)
        
        # Log encoders as MLflow artifacts
        mlflow.log_artifact(f"{tmpdir}/le_item.pkl", "encoders")
        mlflow.log_artifact(f"{tmpdir}/le_tier.pkl", "encoders")
    
    print("✓ Encoders logged as MLflow artifacts")
    print(f"  encoders/le_item.pkl  — maps item_type to integer")
    print(f"  encoders/le_tier.pkl  — maps supplier_tier to integer")

## Inference Example

Demonstrates how to load the registered model and encoders from MLflow and generate a prediction on new data — without retraining.

**Sample prediction:**

| Field | Value |
|-------|-------|
| Item type | BEER |
| Supplier tier | top3 |
| Lag 1 sales | $85,000.00 |
| Lag 2 sales | $78,000.00 |
| Rolling 3m avg | $80,000.00 |
| Month | January 2025 |
| **Predicted next month sales** | **$57,575.43** |

The model predicts $57,575.43 for the following month. The forecast is conservative relative to the $80,000 rolling average — reflecting the slightly declining trend in the two lag features ($85,000 → $78,000). The model interprets that downward momentum as a signal that sales will continue to decrease in the next period.

This is the minimum viable inference pipeline for production use. Any new monthly record with the same 10 features can be passed through this code to get a sales forecast.

In [0]:
#   CELL 13 — INFERENCE EXAMPLE

import mlflow.pyfunc
import pickle

# Load the registered model from MLflow Model Registry
model_uri = "models:/workspace.default.lightgbm_sales_forecaster/2"
loaded_model = mlflow.pyfunc.load_model(model_uri)

print("✓ Model loaded from MLflow Model Registry")

# Load encoders directly from the run artifacts
client         = mlflow.tracking.MlflowClient()
model_versions = client.search_model_versions("name='workspace.default.lightgbm_sales_forecaster'")
run_id         = model_versions[0].run_id

tmp_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="encoders")

with open(f"{tmp_path}/le_item.pkl", "rb") as f:
    le_item_loaded = pickle.load(f)

with open(f"{tmp_path}/le_tier.pkl", "rb") as f:
    le_tier_loaded = pickle.load(f)

print("✓ Encoders loaded from MLflow artifacts")

# Build a sample new record for inference
# This represents: BEER, top3 supplier, January 2025
new_data = pd.DataFrame([{
    "item_type"        : "BEER",
    "supplier_tier"    : "top3",
    "lag_1_sales"      : 85000.0,
    "lag_2_sales"      : 78000.0,
    "rolling_3m_avg"   : 80000.0,
    "warehouse_ratio"  : 0.65,
    "transaction_count": 450,
    "month_sin"        : 0.5,
    "month_cos"        : 0.866,
    "year"             : 2025
}])

# Apply encoders to categorical columns
new_data["item_type_enc"]     = le_item_loaded.transform(new_data["item_type"])
new_data["supplier_tier_enc"] = le_tier_loaded.transform(new_data["supplier_tier"])

# Select only the feature columns used during training
new_data_features = new_data[FEATURE_COLS]

# Generate prediction
prediction = loaded_model.predict(new_data_features)

print(f"\nInput record:")
print(f"  Item type      : BEER")
print(f"  Supplier tier  : top3")
print(f"  Lag 1 sales    : $85,000.00")
print(f"  Lag 2 sales    : $78,000.00")
print(f"  Rolling 3m avg : $80,000.00")
print(f"  Month          : January 2025")
print(f"\nPredicted next month sales : ${prediction[0]:,.2f}")

## Results & Conclusions

### Best model: LightGBM

Although XGBoost achieved the lowest CV MAE of $116.80, LightGBM was selected as the best model based on test set performance — the only evaluation that reflects real-world behavior on unseen data.

| Metric | Value |
|--------|-------|
| Test MAE | $276.81 |
| Test RMSE | $1,312.87 |
| Test R² | 96.84% |

On average, the model's monthly sales predictions are off by **$276.81**. It explains **96.84%** of the variance in sales across all supplier and product combinations.

### Model ranking: best to worst

**1. LightGBM — best overall**
Wins on Test MAE ($276.81), Test RMSE ($1,312.87), and Test R² (96.84%). Although its CV MAE of $255.03 looks worse than XGBoost and Random Forest during validation, it generalizes the best to unseen data. That is what matters in production.

**2. Random Forest — most consistent**
Second best on Test MAE ($284.64) and Test R² (95.70%). Its CV MAE ± of $57.73 is the lowest of all models, meaning it performs consistently across different time periods. A reliable choice when stability matters more than peak accuracy.

**3. XGBoost — overfits slightly**
Best CV MAE of $116.80 during validation, but Test MAE jumps to $305.23 on 2020 data — a gap of $188.43. That difference between validation and test performance is a sign of mild overfitting. The model learned the training distribution too well and struggled to adapt to the demand shock of 2020.

**4. Linear Regression — baseline only**
Worst performance across all metrics: Test MAE of $359.67, Test RMSE of $2,038.51, and R² of 92.39%. Its CV MAE ± of $245.58 also reveals high instability across folds. It served its purpose as a baseline — confirming that the tree-based models provide meaningful improvement — but it is not suitable for production use.

### What drives predictions

The two lag features (`lag_1_sales` and `lag_2_sales`) account for 71% of all model decisions. Recent sales history is by far the strongest signal — the model is essentially learning that next month will look like the last two months, adjusted by supplier size and trend. `supplier_tier` ranks third at 13.76%, ahead of `rolling_3m_avg` at 11.34%. Seasonality, product type, and warehouse ratio contribute less than 5% combined.

### Error analysis by segment

**By product type:** The model is production-ready for LIQUOR (MAE $172.84, R² 95.42%) and WINE (MAE $145.36, R² 90.77%). BEER has the highest MAE in dollar terms ($907.23) but the strongest R² (96.88%) — large errors are driven by the COVID-19 demand shock in March 2020, not by systematic model failure. KEGS has a negative R² (-0.0418), meaning the model performs worse than simply predicting the average — keg demand is driven by events and seasonal patterns that lag features cannot capture. NON-ALCOHOL shows R² of 49.68%, indicating the current feature set is insufficient for that segment.

**By supplier tier:** `rest` suppliers have the lowest MAE in dollar terms ($112.22) due to low and stable sales volumes. `top3` suppliers have the highest MAE ($3,160.43) but the strongest R² (97.27%) — absolute errors are large because their sales volumes are large, but the underlying pattern is well captured. `top15` has the weakest R² (87.51%), making it the hardest segment to forecast reliably.

### Monthly error pattern (2020)

The model underestimates sales in January (-15.41%) and March (-12.79%), and overestimates in July (+7.31%). January and March errors reflect the COVID-19 demand shock — a surge in anticipatory buying followed by abrupt business closures. The July overestimation suggests the model expected a recovery that did not fully materialize, as many businesses remained partially closed through mid-2020.

### Honest assessment of test results

Test metrics are weaker than CV metrics across all models. This is expected and intentional — the test set covers 2020, a year disrupted by the COVID-19 pandemic. No model trained on 2017–2019 data could reasonably anticipate that demand shock. These metrics reflect honest performance under real-world conditions, not ideal ones.

### Limitations

- The model has no external signals (promotions, holidays, economic indicators) that could improve accuracy
- KEGS and NON-ALCOHOL segments require additional features such as event calendars or category-specific seasonality indicators to forecast reliably
- Hyperparameters were selected based on established best practices for gradient boosting on tabular data — no exhaustive grid search was performed
- Predictions for suppliers with fewer than 6 months of history are excluded by design
- The model is sensitive to external demand shocks (pandemics, economic crises) that fall outside the distribution of the training data